# snowseer — analysis notebook

> *Constants as the bridge.* Generalisation, not memorisation.
> SoTA Commission I — Minimal-Shot Autonomy.

The brief asks for a notebook that walks through *the thinking, not just the final pipeline*. That is what this notebook tries to be — exploration, decisions, the dead ends that shaped the system, the integrity audit, and the honest limits — rather than a polished restatement of the writeup.

There are no IoU / coverage / accuracy numbers reported here. The project has no pixel-level snowy-road ground truth (Boreas's cm-accurate Applanix poses don't address that gap), so any quoted metric would be cherry-picked. The honest claim is qualitative, and the qualitative claim is what this notebook works towards.

The headline artefact lives at `outputs/video/boreas_2021_01_26/overlay.mp4`. Reproduce with `make reproduce`. This notebook focuses on the static demo to keep individual cells inspectable; the moving demo is the same recipe in a track-iterating wrapper.


## 1 · Where we started

The project began as a static-image experiment: take a snowy street photograph, take a clear-season photograph of the same place, see if a frozen feature matcher could find correspondences across the season change, and if a homography fitted to those correspondences could carry a Cityscapes-trained road segmenter's output from the clear image into the snow image.

What we *expected* to be hard:

- DISK + LightGlue handling cross-season appearance variation (lighting, foliage, lens, snow on surfaces). MegaDepth-trained matchers can be brittle on out-of-distribution domain pairs.
- Mask2Former's road class behaving sensibly across the long tail of European-vs-North-American urban contexts (Cityscapes is European-trained).

What turned out to *actually* be hard:

- The geometry. A single planar homography is exact only for a planar scene; buildings and trees aren't planar. The matcher would happily find correspondences on building façades, RANSAC would happily fit those, and the resulting homography would warp the *road* mask onto a façade. Solving this empirically (ground-plane biasing the RANSAC) drove a substantial fraction of the project effort.
- The motion artefacts. Several "obvious improvements" that look great in single-frame stills are catastrophic in video. The discipline that emerged — *do not call a video result a win from sampled stills* — applies whenever a quality measure is computed per-frame. This notebook documents three of those.

What was *easier than expected*:

- Cross-season matching itself. DISK + LightGlue, frozen, finds enough correspondences on gate posts, fence wires, masonry corners, and rooflines to fit a usable homography most of the time. The matcher's "anchor on what doesn't change" property is observable, not designed; we did not have to engineer it.


## 2 · Data inspection

We load one canonical (snow, clear) pair from the demo set. The two images are taken at the same physical place in different seasons. The matcher's job is to find what the season has not changed.


In [ ]:
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists():
            return p
    raise RuntimeError(f'Could not find pyproject.toml above {cur}')

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import cv2
import matplotlib.pyplot as plt

# Pick the first demo pair on disk that has both snow and clear images.
# Run `make stills` first if no pairs are present (fresh clones).
candidates = sorted((ROOT / 'data/pairs').iterdir()) if (ROOT / 'data/pairs').is_dir() else []
PAIR_DIR = next(
    (p for p in candidates if p.is_dir() and (p / 'snow.jpg').exists() and (p / 'clear.jpg').exists()),
    None,
)
if PAIR_DIR is None:
    raise RuntimeError(
        f'No demo pairs found under {ROOT}/data/pairs. '
        'Run `make stills` (needs MAPILLARY_TOKEN) from the repo root, '
        'then re-execute this notebook.'
    )
print(f'using pair: {PAIR_DIR.name}')

snow = cv2.cvtColor(cv2.imread(str(PAIR_DIR / 'snow.jpg')), cv2.COLOR_BGR2RGB)
clear = cv2.cvtColor(cv2.imread(str(PAIR_DIR / 'clear.jpg')), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(snow);  axes[0].set_title('Snow query');  axes[0].axis('off')
axes[1].imshow(clear); axes[1].set_title('Clear-season prior'); axes[1].axis('off')
plt.tight_layout(); plt.show()


The two regimes look different. The road in the snow image is a flat white surface; in the clear prior, the road has lane markings, kerbs, sometimes wet asphalt under cloud. Buildings, fences, gate posts, the corner of a red house, signposts — those are present in both, in roughly the same image positions when the camera was roughly in the same physical pose. Those are the constants the matcher can anchor on.


## 3 · First match attempt — what does DISK + LightGlue actually anchor on?

We run DISK + LightGlue across the pair without any custom training, then plot the inlier subset of correspondences after RANSAC. The instructive part is *where the lines land*.


In [ ]:
from src.matching import Matcher, draw_matches
from src.homography import estimate

matcher = Matcher()
match_result = matcher.match(snow, clear)
homo = estimate(match_result, snow.shape[:2], clear.shape[:2])

canvas = draw_matches(
    snow, clear, match_result,
    inlier_mask=homo.inlier_mask,
    max_inliers=25, max_outliers=0,
)
plt.figure(figsize=(14, 6)); plt.imshow(canvas); plt.axis('off')
plt.title('Snow ↔ clear correspondences (25-inlier subset)')
plt.show()


The lines connect features the season has not changed: gate posts, fence wires, masonry corners, the corner of a roof, a window frame. There are no correspondences on the road surface itself — the snow has erased the lane markings and the kerb, and the matcher knows there is nothing to anchor on there.

That is fine, because we are not trying to match road to road. We are using the constants on the buildings and fences to fix the *geometry*, then using the geometry to bring the *clear* image's road mask back to the snow image's pixel space. The matcher's behaviour here — anchoring on what doesn't change between regimes — is the property the brief calls *generalisation, not memorisation*. We did not design that property in; the matcher came pretrained on MegaDepth and we left it untouched.


## 4 · Why ground-plane-biased homography

A single homography is exact only for a planar scene. The world is not planar — buildings, trees, signposts, and pedestrians sit at varying depths off the road plane. RANSAC, given enough correspondences on those off-plane features, will happily fit a homography that fits the *façade* plane rather than the *road* plane, and warp the road mask onto the side of a building.

Empirically, restricting RANSAC to correspondences in the *lower* half of both images (`y > 0.5 H`) biased the fit toward the ground plane reliably enough that the warp transports the road, not the buildings. This is implemented in `src/homography.estimate` as `ground_plane_y_frac=0.5`. A complementary `dashboard_y_frac` cap (default 1.0; lower values for vehicles with prominent dashboards in the bottom of the frame) drops correspondences in the dashboard band that would otherwise dominate the lower-image sample.

The combined effect is that off-plane mismatches concentrate where they are visible (façades) and where we can ignore them (the upper half of the image, which we don't claim as drivable anyway).


## 5 · The full single-frame pipeline

Match → estimate homography → segment the *clear* prior → warp the mask back into snow space → overlay. End-to-end on one frame, qualitative.


In [ ]:
from src.segmentation import RoadSegmenter
from src.overlay import warp_mask, alpha_blend, keep_largest_component

seg = RoadSegmenter()
road_mask_clear = seg.segment_road(clear)
road_mask_clear = keep_largest_component(road_mask_clear)

H_inv = np.linalg.inv(homo.H)
road_mask_snow = warp_mask(road_mask_clear, H_inv, snow.shape[:2])
road_mask_snow = keep_largest_component(road_mask_snow)

overlay = alpha_blend(snow, road_mask_snow, color=(46, 156, 86), alpha=0.50)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(alpha_blend(clear, road_mask_clear, color=(46, 156, 86), alpha=0.50))
axes[0].set_title('Clear prior + Mask2Former road'); axes[0].axis('off')
axes[1].imshow(canvas); axes[1].set_title('Correspondences'); axes[1].axis('off')
axes[2].imshow(overlay); axes[2].set_title('Snow + warped road overlay'); axes[2].axis('off')
plt.tight_layout(); plt.show()


The plough now knows where the road is. Mask2Former — frozen on Cityscapes, never shown a snowy frame — produced the green mask on the clear prior. The homography estimated from constants visible in both seasons brought it onto the snow frame. No model in the pipeline ran on a snowy image during training.


## 6 · Scaling to video — three thin layers

The single-frame demo is one frame. The video extension wraps the same per-pair pipeline in three thin layers, each justified by a single per-frame need:

- **Track loader** (`src/video_runtime/track.py`). The snow stream and the paired summer stream both come with `(GPSTime, easting, northing, heading)` per frame. The loader indexes both so a snow frame's pose can be looked up and matched against summer poses in metric distance.
- **Prior pool** (`src/video_runtime/prior_pool.py`). For each snow frame, return the K = 3 nearest summer captures by UTM distance. Mask2Former runs once per *unique* summer prior — its mask is cached and re-used across snow frames that select the same prior. K = 3 instead of K = 1 because per-prior aliasing on the warped boundary is suppressed by averaging across multiple priors.
- **EMA temporal smoother** (`src/video_runtime/temporal.EMASmoother`, $\\alpha = 0.4$). Suppresses per-frame jitter; on a frame where the matcher fails entirely, the previous smoothed mask is held — graceful degradation rather than an empty-frame flicker. The reasons for *not* using more sophisticated smoothing are the next three sections.

A pickled cache layer (`outputs/video/<track>/_cache_<tag>.pkl`) stores the per-frame `FrameResult` so subsequent renders that change only the smoother or the layout skip the matching pass entirely. The matching pass dominates per-frame compute; everything downstream is cheap.


## 7 · Dead end #1 — synthetic priors from past frames

The reasoning was sound. Snow → snow matching has identical lighting, lens, and viewpoint conditions between consecutive snow frames, so DISK + LightGlue returns roughly three to seven times more correspondences per pair than snow → summer. In a single-frame still, the warped-back mask from a synthetic past-frame prior was visibly broader and more confident than the mask from a summer-prior fit.

In motion, this fails in a way the static figure cannot show. Each frame's slightly-too-large mask seeded the *next* frame's synthetic prior, which produced a slightly-larger mask, which seeded a still-larger mask, and so on. Over five to ten seconds the road overlay drifted outward into bushes and treelines. A positive-feedback loop in which higher matcher confidence reflected matching to an *increasingly wrong* reference.

We rejected synthetic priors. The lesson generalises: a smoothing or augmentation strategy that recursively feeds its own output back into the matcher is dangerous in motion even when it looks correct per-frame. The observable distinction is the loop, not the per-frame quality.


## 8 · Dead end #2 — optical-flow propagation

The motivation for trying optical flow was speed. Matching is the dominant per-frame cost; if dense optical flow could *propagate* a previously-matched mask between sparsely-matched keyframes, the matching cost would amortise and the system might be fast enough for live operation.

The implementation worked per-frame. The motion failure was the same shape as synthetic priors, with a different mechanism: a forward-driving camera has *vanishing-point flow*, and dense flow remap-reuses last-frame pixels along the radial direction. Each propagation step pushed mask boundaries outward at every step. The road region grew until it engulfed the sidewalks and then the verge.

Same shape as #1: the flow-propagated mask is the next frame's seed; the next frame's seed is wrong; the next-next frame's seed is more wrong. The fact that the failure shape repeated even when the *mechanism* was different (vanishing-point flow vs. synthetic-prior identity) is what made us suspect the deeper rule — *positive feedback in mask propagation through motion is the failure pattern, not the specific algorithm.*

We rejected optical-flow propagation. We kept a binary-mask EMA at $\\alpha = 0.4$ — the simplest possible smoother, and the one that does the least damage *because* it does not feed its output back into the matcher.


## 9 · Dead end #3 (smaller) — the inlier-count surprise

This one is less a dead end and more a piece of received wisdom we caught misleading us. We had been ranking demo pairs by RANSAC inlier count, on the intuition that more correspondences = more confident matching = better fit. The static-stills curator (`make stills` audit step) did the same.

The audit on twenty-seven pairs produced a counterexample we should have anticipated. A pair with >100 inliers can produce a *worse* overlay than a pair with 30 inliers if the 100 inliers concentrate on a couple of building façades. RANSAC is fitting the façade plane, and the road mask is being warped onto a wall. Conversely, a pair with 30 inliers spread across the lower half of the image (gate posts, kerb-line markings on the prior, distant rooflines) produces a clean ground-plane fit.

The lesson: *inlier count is a property of the matcher's confidence, not of the geometry it fits.* The geometric quality is determined by where the inliers *live in the image*, not how many there are. Practical consequence: the system needs human review on input (is the (snow, clear) pair plausibly the same place from the same vantage?) and on output (is the warped road overlay actually on the road, not on a façade?), even after RANSAC succeeds with a high inlier count.

The static-stills audit pipeline (`src/audit.py`) generates a 4-panel contact sheet per pair specifically so a human can do that review at a glance.


## 10 · Minimal-shot integrity audit

The brief asks for systems that *generalise* into unseen regimes rather than systems that *memorise* the regimes they were trained on. The contribution is honest only if no snowy data ever entered the training pipeline of any learned component. We codify this as a checklist; the cell below confirms it programmatically by importing each component and listing its training corpus.


In [ ]:
import importlib

COMPONENTS = [
    ('DISK',         'MegaDepth (outdoor scenes; no snow split)',
                     'src.matching', 'Matcher'),
    ('LightGlue',    'MegaDepth (matched DISK features)',
                     'src.matching', 'Matcher'),
    ('Mask2Former',  'Cityscapes (clear-weather European driving; no snow)',
                     'src.segmentation', 'RoadSegmenter'),
]

print(f"{'Component':14s}  {'Training corpus':50s}  Snow in training?")
print('-' * 95)
for name, corpus, mod_path, cls_name in COMPONENTS:
    mod = importlib.import_module(mod_path)
    assert hasattr(mod, cls_name), f'{cls_name} missing from {mod_path}'
    print(f"{name:14s}  {corpus:50s}  no")
print()
print('USAC-MAGSAC: classical RANSAC, no learning. No training corpus.')
print()
print('Snow imagery enters the pipeline only as a runtime input to Matcher.match()')
print('and overlay.warp_mask(). It never participates in any model parameter update.')


## 11 · Where the design landed

Three short paragraphs on the choices that shaped the system. None are quantified, because none of the trade-offs decompose cleanly into a number we can ground-truth against.

**K = 3 priors per snow frame, not K = 1 or K = 5.** With K = 1, per-prior boundary aliasing in the warp shows up as flicker on the mask edge in motion. With K = 5, the additional priors come from progressively further away in UTM distance and contribute lower-quality matches that dilute the average. K = 3 was the empirically-stable sweet spot — close enough that all three priors are within a few metres of the snow pose, far enough apart that boundary-aliasing artefacts decorrelate.

**Soft-average fusion weighted by inlier count, not majority vote or argmax.** Majority vote is brittle when one prior fails (a 2/3 majority becomes a 1/2 tie); argmax-by-confidence picks the single best prior and discards the others' information. Soft average gracefully degrades — a prior with weak matches contributes little weight, a prior with strong matches dominates, the boundary is smoother than any single prior's. This is `src/fuse.weighted_soft_average`.

**EMA on the binary mask at $\\alpha = 0.4$, not flow-propagation, not Kalman-on-mask, not homography smoothing.** EMA is the simplest available smoother. It does the least damage. It does not feed its output back into the matcher. On a frame where matching fails entirely, it holds the previous good mask rather than producing an empty mask. We tried two more sophisticated alternatives (sections 7 and 8) and they failed in motion in ways the static demo hid; we kept EMA because *simplicity that does not recursively amplify error* turned out to matter more than per-frame fidelity.


## 12 · Where this could go

The constants-bridge primitive is the foundation for a more general image-banking-and-transfer appliance. Live frame plus location signal; bank of geo-tagged clear-conditions imagery; register the live frame against nearest bank candidates via geometric correspondence; transfer any pre-computed annotation from the bank into live-frame coordinates.

The snowplough's road-position channel is one consumer of this appliance. The same recipe and the same code paths power *fog / dust / smoke / heavy rain / night driving* (the bank is the clear-conditions capture of the same place), *seeing round corners* (the bank is earlier captures of the upcoming intersection — from this drive or from V2V partners), *heads-up display navigation* (the bank is street-view), *construction-zone delta detection* (bank vs. live = what's new on site), and *industrial inspection in obscured environments* (mining / post-fire / decommissioning — pre-incident facility scan as the bank).

Concrete code surfaces that change as the project moves toward the appliance:

- `src/matching.Matcher` becomes pluggable for a denser, GPU-resident alternative (RoMa, LoFTR, DKM). The current DISK + LightGlue stack is matcher-bottlenecked at ~16 s per frame on CPU; a dense alternative on GPU is the path to ~30 ms.
- `src/video_runtime/prior_pool.PriorPool` gains a *visual place recognition* front-end so the appliance works without prior pose. Currently the lookup is GPS-keyed.
- The substrate-pluggable layer in `_archive/v3_substrate_experiments/` (which did not beat operator-paired drives in our v3 experiment because the matcher couldn't bridge cross-camera mismatch) becomes viable again with a denser cross-domain matcher.
- A small embedded entry point — Jetson-class — runs the live appliance with a HUD output, demonstrating snow / fog / night-driving consumers end to end.
- Two more constants-bridge instances added as worked examples: agricultural autonomy (drone overflight bank → ground vehicle) and industrial inspection in obscured environments.


## 13 · Honest limits, honest claims

The output answers *where the road should be*, not *where to drive*. A snow-covered car parked on the road would still sit inside the green overlay; the system has no notion of obstacles, drivable surface, or 3D geometry. The pipeline contributes a 2-D road-position prior to a fuller perception stack alongside lidar, depth, and obstacle detection. It does not replace any of them.

Within that scope, the qualitative claim is what the project actually demonstrates: *the road overlay tracks the buried road continuously through the canonical clip on a pipeline whose learned components have never seen snow*, and *the same pipeline runs unchanged on a different snowfall on the same loop and on twenty-seven static pairs in different countries*. We do not quote IoU or coverage percentages because we have no labelled ground truth; quoted numbers without ground truth are cherry-picked by definition. The system needs human review on input (is the (snow, clear) pairing plausibly the same place from the same vantage?) and on output (is the overlay actually on the road, not on a façade?) even after the matcher succeeds.

The pipeline is *not real-time* on Mac CPU. The matcher dominates per-frame compute; the canonical 15-second clip's cache builds in roughly an hour. The cache layer makes that cost amortise across renders, but real-time on a vehicle would require a substantially faster correspondence model — a learned component, ideally trained on synthetic snow + clear pairs to preserve the no-snow-in-training guarantee. This is a deployment-engineering question, not a research one, and is the first item in section 12.

---

*Code, video clips, and the static-stills precursor: see the [project repository](../README.md). Submitted to SoTA Commission I — Minimal-Shot Autonomy, May 2026.*
